# DHL Shipment Tracking & Logistics Optimization RAG System
## Using LangGraph + OpenAI GPT API + Vector Database

**Objective:** Build a RAG (Retrieval-Augmented Generation) system that answers DHL-related shipment tracking and logistics optimization queries using LangGraph agents.

**Architecture:**
- Vector Database (Chroma) - Store DHL knowledge
- LLM (OpenAI GPT) - Generate intelligent responses
- LangGraph - Orchestrate multi-step agent workflows
- Sample Data - Synthetic DHL shipments & routes


In [ ]:
# Step 1: Install Required Libraries
import subprocess
import sys

packages = [
    "langchain",
    "langgraph",
    "langchain-openai",
    "langchain-chroma",
    "langchain-text-splitters",
    "langchain-huggingface",
    "pandas",
    "numpy",
    "python-dotenv"
]

print("Installing required packages...")
for package in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

print("✅ All packages installed successfully!")
print("\n📌 Next Step: Set your OPENAI_API_KEY environment variable")

In [ ]:
# Step 2: Verify All Packages Installation
import subprocess
import sys

packages = [
    "langchain",
    "langgraph",
    "langchain_openai",
    "langchain_chroma",
    "langchain_text_splitters",
    "langchain_huggingface",
    "pandas",
    "numpy",
    "dotenv"
]

print("🔍 Checking installed packages...\n")

all_ok = True
for package in packages:
    try:
        __import__(package)
        print(f"✅ {package:<25} - OK")
    except ImportError:
        print(f"❌ {package:<25} - MISSING (reinstalling...)")
        all_ok = False
        pip_name = package.replace("_", "-")
        if pip_name == "dotenv":
            pip_name = "python-dotenv"
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])
        print(f"   └─ Reinstalled successfully")

print("\n" + "="*50)
if all_ok:
    print("✅ All packages are properly installed!")
else:
    print("⚠️ Some packages were missing. Reinstalled them.")
print("="*50)

In [ ]:
# Step 2.5: Setup OpenAI API Key
import os
from dotenv import load_dotenv

load_dotenv()

# Check for API key
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY:
    print("OPENAI_API_KEY found")
    print(f"Key starts with: {OPENAI_API_KEY[:10]}...")
else:
    print("OPENAI_API_KEY not found")
    print("\nSetup Instructions:")
    print("Option 1: Create a .env file with:")
    print("OPENAI_API_KEY=sk-...your-key...")
    print("")
    print("Option 2: Set environment variable directly:")
    print("os.environ['OPENAI_API_KEY'] = 'your-key-here'")
    print("")
    print("Get your API key from the OpenAI Platform API keys page.")

    # Uncomment and add your key here if needed:
    # os.environ["OPENAI_API_KEY"] = "your-key-here"

print("\nUsing OpenAI GPT API")
print("Model: gpt-4.1-mini")


In [ ]:
# Step 3: Import Libraries & Setup
import os
from typing import Optional
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# LangChain & LangGraph imports - using OpenAI GPT API
from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain.tools import tool
from langgraph.graph import StateGraph
from langgraph.types import Command
from typing_extensions import TypedDict
import json
import pandas as pd

print("✅ Libraries imported successfully")
print("📌 Using OpenAI GPT API with HuggingFace embeddings")

In [ ]:
# Step 3.1: RAG Configuration
from pathlib import Path

# Toggle data source (synthetic by default for demo)
USE_SYNTHETIC_DATA = False

# Text splitting
CHUNK_SIZE = 500
CHUNK_OVERLAP = 100

# Vector store
COLLECTION_NAME = "dhl_knowledge"
PERSIST_DIR = "chroma_db"

# Retrieval
TOP_K = 3
USE_MMR = True
FETCH_K = 20
LAMBDA_MULT = 0.5

print("RAG config loaded")
print(f"- chunks: {CHUNK_SIZE} / overlap {CHUNK_OVERLAP}")
print(f"- retriever: {'mmr' if USE_MMR else 'similarity'}, top_k={TOP_K}")
print(f"- vector store: {COLLECTION_NAME} @ {PERSIST_DIR}")


In [ ]:
# Step 3.2: Optional CSV Loader (for real data later)
def load_knowledge_from_csv(path: str) -> list[Document]:
    """
    Expected columns: content (required), source, type, id
    """
    df = pd.read_csv(path)
    if "content" not in df.columns:
        raise ValueError("CSV must include a 'content' column")

    docs = []
    for i, row in df.iterrows():
        metadata = {
            "source": row.get("source", Path(path).name),
            "type": row.get("type", "doc"),
            "id": row.get("id", f"doc_{i:04d}"),
        }
        docs.append(Document(page_content=str(row["content"]), metadata=metadata))

    return docs

print("CSV loader ready (set USE_SYNTHETIC_DATA = False to use)")


In [29]:
# Step 4: Create Sample DHL Knowledge Base
if USE_SYNTHETIC_DATA:
    dhl_knowledge_base = [
        Document(
            page_content="DHL Express Shipment Service: Premium overnight delivery service. Standard delivery within 24-48 hours for domestic shipments. International shipments available to 220+ countries. Tracking available via website or mobile app.",
            metadata={"source": "DHL Services", "type": "service", "id": "svc_express"}
        ),
        Document(
            page_content="Shipment Tracking Status Codes: IN_TRANSIT (on the way), OUT_FOR_DELIVERY (with local delivery partner), DELIVERED (successfully delivered), EXCEPTION (delay or issue), RETURNED (sent back to sender). Updates available real-time.",
            metadata={"source": "DHL Documentation", "type": "tracking", "id": "tracking_codes"}
        ),
        Document(
            page_content="DHL Standard Service SLA: Domestic shipments deliver in 2-3 business days. Express service delivers in 1 business day. Weekend and holiday delivery available in select locations. Peak season (Nov-Dec) may add 1-2 days.",
            metadata={"source": "DHL SLA", "type": "service_level", "id": "sla_standard"}
        ),
        Document(
            page_content="Route Optimization Best Practices: Group shipments by destination zone. Use regional hubs for efficiency. Peak hours: 9-11 AM, 2-4 PM. Avoid peak hours for non-urgent shipments. Use consolidated shipments for cost savings (15-25% discount).",
            metadata={"source": "Logistics Optimization", "type": "routing", "id": "routing_best_practices"}
        ),
        Document(
            page_content="DHL Pricing Model: Base rate starts at $5.99 for standard 2-3 day delivery. Express 1-day: $14.99. International: varies by destination. Weight surcharge: +$0.50 per lb over 1 lb. Weekend delivery: +$3.00. Volume discounts available at 100+ shipments/month (10-20% off).",
            metadata={"source": "DHL Pricing", "type": "pricing", "id": "pricing_model"}
        ),
        Document(
            page_content="Problem Resolution Process: Step 1 - Check tracking status (may resolve 70% of inquiries). Step 2 - Contact DHL support if status shows EXCEPTION. Step 3 - File claim if package is lost (up to $100 for standard, $500 for express). Step 4 - Insurance claim for items over $500.",
            metadata={"source": "Support Procedures", "type": "support", "id": "support_process"}
        ),
        Document(
            page_content="Logistics Network: DHL operates 350+ service centers globally. Average processing time: 2 hours per facility. Cross-dock operations reduce handling. Peak season: 20-30% slower processing. Summer (May-Aug): fastest processing times.",
            metadata={"source": "Network Operations", "type": "operations", "id": "network_ops"}
        ),
    ]

    print(f"Created {len(dhl_knowledge_base)} synthetic knowledge base documents")
    for doc in dhl_knowledge_base[:3]:
        print(f"- {doc.metadata['source']} ({doc.metadata['id']}): {doc.page_content[:60]}...")
else:
    dhl_knowledge_base = load_knowledge_from_csv("data/dhl_knowledge.csv")
    print(f"Loaded {len(dhl_knowledge_base)} documents from CSV")


✅ Created 7 knowledge base documents
  - DHL Services: DHL Express Shipment Service: Premium overnight delivery ser...
  - DHL Documentation: Shipment Tracking Status Codes: IN_TRANSIT (on the way), OUT...
  - DHL SLA: DHL Standard Service SLA: Domestic shipments deliver in 2-3 ...


In [ ]:
# Step 5: Setup Vector Database (Chroma) with HuggingFace Embeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Initialize embeddings using HuggingFace (free, runs locally)
# Using HuggingFace embeddings locally to keep the demo self-contained
print("Loading HuggingFace embeddings model...")
print("(First run downloads ~90MB model)")

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Split documents for better retrieval
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP
)

split_docs = text_splitter.split_documents(dhl_knowledge_base)

print(f"Splitting documents... {len(split_docs)} chunks created")

# Create / Load Chroma vector store (persisted)
persist_path = Path(PERSIST_DIR)
db_exists = (persist_path / "chroma.sqlite3").exists()

if db_exists:
    vector_store = Chroma(
        collection_name=COLLECTION_NAME,
        embedding_function=embeddings,
        persist_directory=PERSIST_DIR
    )
    print(f"Loaded existing Chroma DB from {PERSIST_DIR}")
else:
    vector_store = Chroma.from_documents(
        documents=split_docs,
        embedding=embeddings,
        collection_name=COLLECTION_NAME,
        persist_directory=PERSIST_DIR
    )
    if hasattr(vector_store, "persist"):
        vector_store.persist()
    print(f"Built and persisted Chroma DB at {PERSIST_DIR}")

# Create retriever
if USE_MMR:
    retriever = vector_store.as_retriever(
        search_type="mmr",
        search_kwargs={"k": TOP_K, "fetch_k": FETCH_K, "lambda_mult": LAMBDA_MULT}
    )
else:
    retriever = vector_store.as_retriever(search_kwargs={"k": TOP_K})

print(f"Vector database initialized with {len(split_docs)} chunks")
print("Sample retrieval test:")

# Test retrieval
test_query = "How long does DHL delivery take?"
retrieved_docs = retriever.invoke(test_query)
for i, doc in enumerate(retrieved_docs, 1):
    src = doc.metadata.get("source", "unknown")
    doc_id = doc.metadata.get("id", "n/a")
    print(f"  Result {i}: {doc_id} | {src} | {doc.page_content[:80]}...")


In [ ]:
# Step 5.1: Quick Retrieval Evaluation (Smoke Test)
eval_set = [
    {"query": "What are the standard delivery times?", "expected_ids": ["sla_standard"]},
    {"query": "What does EXCEPTION status mean?", "expected_ids": ["tracking_codes"]},
    {"query": "How are volume discounts applied?", "expected_ids": ["pricing_model"]},
]

def evaluate_retriever(retriever, eval_set):
    hits = 0
    for ex in eval_set:
        docs = retriever.invoke(ex["query"])
        got_ids = {d.metadata.get("id") for d in docs}
        hit = any(eid in got_ids for eid in ex["expected_ids"])
        hits += int(hit)
        print(f"- {ex['query']} -> {'HIT' if hit else 'MISS'} | got {sorted(got_ids)}")

    hit_rate = hits / len(eval_set)
    print(f"Retrieval hit rate: {hit_rate:.0%} ({hits}/{len(eval_set)})")

# Run evaluation
evaluate_retriever(retriever, eval_set)


In [31]:
# Step 6: Create Sample Shipment Data
import pandas as pd
from datetime import datetime, timedelta
import random

# Sample shipments for demonstration
shipments_data = {
    "tracking_id": ["DHL001", "DHL002", "DHL003", "DHL004", "DHL005"],
    "from_city": ["New York", "Los Angeles", "Chicago", "Houston", "Miami"],
    "to_city": ["Boston", "San Francisco", "Detroit", "Dallas", "Atlanta"],
    "status": ["IN_TRANSIT", "OUT_FOR_DELIVERY", "DELIVERED", "IN_TRANSIT", "EXCEPTION"],
    "weight_lb": [2.5, 5.0, 1.2, 8.3, 3.7],
    "shipping_date": [
        datetime.now() - timedelta(days=2),
        datetime.now() - timedelta(days=1),
        datetime.now() - timedelta(days=3),
        datetime.now() - timedelta(days=1),
        datetime.now() - timedelta(days=4),
    ],
    "estimated_delivery": [
        datetime.now() + timedelta(days=1),
        datetime.now() + timedelta(hours=4),
        datetime.now() - timedelta(days=0),
        datetime.now() + timedelta(hours=12),
        datetime.now() + timedelta(days=2),
    ],
    "cost": [12.50, 7.99, 5.99, 18.75, 14.99],
}

shipments_df = pd.DataFrame(shipments_data)

print("✅ Sample Shipment Data:")
print(shipments_df.to_string())

# Store as JSON for retrieval
shipments_json = shipments_df.to_json(orient="records", date_format="iso")
print("\n📦 Shipments available for query")

✅ Sample Shipment Data:
  tracking_id    from_city        to_city            status  weight_lb              shipping_date         estimated_delivery   cost
0      DHL001     New York         Boston        IN_TRANSIT        2.5 2026-02-01 15:26:04.070431 2026-02-04 15:26:04.070441  12.50
1      DHL002  Los Angeles  San Francisco  OUT_FOR_DELIVERY        5.0 2026-02-02 15:26:04.070437 2026-02-03 19:26:04.070442   7.99
2      DHL003      Chicago        Detroit         DELIVERED        1.2 2026-01-31 15:26:04.070438 2026-02-03 15:26:04.070443   5.99
3      DHL004      Houston         Dallas        IN_TRANSIT        8.3 2026-02-02 15:26:04.070439 2026-02-04 03:26:04.070444  18.75
4      DHL005        Miami        Atlanta         EXCEPTION        3.7 2026-01-30 15:26:04.070440 2026-02-05 15:26:04.070445  14.99

📦 Shipments available for query


In [33]:
# Step 7: Create LangGraph Agent Tools
from langchain_core.tools import tool

# Tool 1: Query Knowledge Base
@tool
def query_dhl_knowledge(query: str) -> str:
    """Retrieve DHL information from knowledge base"""
    docs = retriever.invoke(query)

    context_lines = []
    sources = []
    for doc in docs:
        source = doc.metadata.get("source", "unknown")
        doc_id = doc.metadata.get("id", "n/a")
        context_lines.append(f"- {doc.page_content} (source: {source}, id: {doc_id})")
        sources.append(f"{source} [{doc_id}]")

    context = "\n".join(context_lines)
    return f"DHL Knowledge:\n{context}\n\nSources: {', '.join(sources)}"

# Tool 2: Check Shipment Status
@tool
def check_shipment_status(tracking_id: str) -> str:
    """Check the status of a DHL shipment"""
    shipment = shipments_df[shipments_df['tracking_id'] == tracking_id.upper()]

    if shipment.empty:
        return f"Shipment {tracking_id} not found"

    ship = shipment.iloc[0]
    status_detail = f"""
    Shipment: {tracking_id}
    From: {ship['from_city']} -> To: {ship['to_city']}
    Status: {ship['status']}
    Weight: {ship['weight_lb']} lbs
    Cost: ${ship['cost']:.2f}
    Estimated Delivery: {ship['estimated_delivery'].strftime('%Y-%m-%d %H:%M')}
    """
    return status_detail.strip()

# Tool 3: Route Optimization
@tool
def optimize_route(from_city: str, to_city: str, num_shipments: int = 1) -> str:
    """Optimize delivery route and suggest cost savings"""
    base_cost = 5.99
    weight_surcharge = 0.5

    # Volume discount
    if num_shipments >= 100:
        discount = 0.20
    elif num_shipments >= 50:
        discount = 0.15
    elif num_shipments >= 10:
        discount = 0.10
    else:
        discount = 0

    total_cost = base_cost * (1 - discount)
    savings = base_cost * discount * num_shipments

    recommendation = f"""
    Route Optimization:
    Route: {from_city} -> {to_city}
    Shipments: {num_shipments}
    Base Cost/Shipment: ${base_cost:.2f}
    Discount Applied: {discount*100:.0f}%
    Cost/Shipment After Discount: ${total_cost:.2f}
    Total Savings: ${savings:.2f}
    """
    return recommendation.strip()

# Register tools as a list (Mistral can handle this)
tools = [query_dhl_knowledge, check_shipment_status, optimize_route]

print(f"Created {len(tools)} tools for LangGraph agent")
for tool in tools:
    print(f"- {tool.name}: {tool.description}")


✅ Created 3 tools for LangGraph agent
  - query_dhl_knowledge: Retrieve DHL information from knowledge base
  - check_shipment_status: Check the status of a DHL shipment
  - optimize_route: Optimize delivery route and suggest cost savings


In [ ]:
# Step 8: Build LangGraph State & Agent (using OpenAI GPT API)
from langgraph.prebuilt import create_react_agent

# Initialize OpenAI LLM
llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0.7
)

# Create agent with tools
agent = create_react_agent(llm, tools)

print("LangGraph Agent created successfully")
print("Agent Configuration:")
print(f"- Model: gpt-4.1-mini (OpenAI API)")
print(f"- Tools: 3 (query_knowledge, check_status, optimize_route)")
print(f"- Framework: LangGraph ReAct Agent")
print("\nYour RAG system is ready. GPT tool-calling is enabled.")


## System Architecture & Interview Talking Points

### Architecture Components:

1. **Vector Database (Chroma)** - Persistent storage of DHL knowledge embeddings
2. **LLM (OpenAI GPT)** - OpenAI API for intelligent responses
3. **Embeddings (HuggingFace)** - Local sentence-transformers for vector search
4. **Retrieval (MMR)** - Diversity-aware retrieval for less redundancy
5. **Tools** - Query knowledge, check status, optimize routes
6. **LangGraph Agent** - ReAct agent that plans and executes actions

### Key Features to Discuss in Interview:

- **RAG Pattern**: Retrieval + generation with source-aware responses
- **Chunking Strategy**: `chunk=500`, `overlap=100` for recall vs. cost tradeoff
- **Persistence**: Reusable Chroma DB (faster re-runs)
- **Evaluation**: Quick retrieval hit-rate smoke test
- **Tool Integration**: Agent uses tools to access structured data
- **Extensibility**: Easy to add APIs, rerankers, or domain filters

### How to Extend for Production:

1. Replace synthetic data with real DHL API integration
2. Add database for storing shipping history
3. Implement caching + request batching
4. Add authentication and rate limiting
5. Deploy as REST API or web service
6. Monitor API costs and add cost optimization
7. Add test suite and error handling


In [36]:
# Interactive Query Function
SYSTEM_PROMPT = """
You are a DHL logistics RAG assistant.
- Use tools when helpful.
- If you use DHL Knowledge, cite sources like [Source|id].
- If information is missing or unclear, say you don't know.
""".strip()

def ask_dhl_agent(user_query: str) -> str:
    """
    Ask the DHL RAG agent any question about shipments and logistics.

    Example questions:
    - "What's the status of shipment DHL002?"
    - "How much would 500 shipments from Chicago to Detroit cost?"
    - "What tracking statuses mean the shipment is delayed?"
    """
    print(f"\nYour Query: {user_query}")
    print("-" * 60)

    response = agent.invoke({
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_query}
        ]
    })

    final_message = response["messages"][-1]
    print(f"Agent Response:\n{final_message.content}")
    return final_message.content

# Example: Customize this with your own query
custom_query = "What are the main factors that affect DHL delivery times, and how can we speed up our shipments?"
ask_dhl_agent(custom_query)



📤 Your Query: What are the main factors that affect DHL delivery times, and how can we speed up our shipments?
------------------------------------------------------------
🤖 Agent Response:
 To speed up DHL deliveries, consider these key factors:

1. **Packaging**: Properly packaging your items ensures they're secure during transit, reducing the chances of damage or delays due to mishandling. Use sturdy boxes and padding materials to protect your goods.
2. **Documentation**: Accurate and complete documentation helps DHL process shipments more efficiently. Make sure all paperwork is correct and up-to-date, including customs declarations and necessary permits.
3. **Shipment size**: Larger shipments may take longer to process and deliver due to their volume. If possible, split larger orders into multiple smaller shipments to expedite the delivery process.
4. **Customs clearance**: International shipments often require customs clearance, which can add time to your delivery schedule. Famil

' To speed up DHL deliveries, consider these key factors:\n\n1. **Packaging**: Properly packaging your items ensures they\'re secure during transit, reducing the chances of damage or delays due to mishandling. Use sturdy boxes and padding materials to protect your goods.\n2. **Documentation**: Accurate and complete documentation helps DHL process shipments more efficiently. Make sure all paperwork is correct and up-to-date, including customs declarations and necessary permits.\n3. **Shipment size**: Larger shipments may take longer to process and deliver due to their volume. If possible, split larger orders into multiple smaller shipments to expedite the delivery process.\n4. **Customs clearance**: International shipments often require customs clearance, which can add time to your delivery schedule. Familiarize yourself with any regulations or requirements in the destination country and provide all necessary documents to minimize delays.\n5. **Choosing the right shipping service**: DHL

## Interactive Query Interface

Use the cell below to ask any custom questions about DHL shipments, logistics, or optimization strategies.

In [40]:
# Test Query 4: Problem Resolution
print("\n" + "=" * 60)
print("QUERY 4: Problem Resolution & Support")
print("=" * 60)

query_4 = "One of our shipments (DHL005) shows an EXCEPTION status. What should we do to resolve this issue and what are our options?"

print(f"\n❓ Query: {query_4}\n")

response = agent.invoke({
    "messages": [
        {"role": "user", "content": query_4}
    ]
})

final_message = response["messages"][-1]
print("🤖 Agent Response:")
print(final_message.content)


QUERY 4: Problem Resolution & Support

❓ Query: One of our shipments (DHL005) shows an EXCEPTION status. What should we do to resolve this issue and what are our options?

🤖 Agent Response:
 Based on the DHL Knowledge provided, here are some steps you can take to resolve the EXCEPTION status for shipment DHL005:

1. Check the shipment details: Verify that all required information such as the sender and recipient addresses, customs declarations, and package contents are accurate.

2. Contact DHL Customer Service: Reach out to DHL's customer service team for a detailed explanation of the exception status and any necessary actions required to resolve it. You can find their contact details on the official DHL website or through your DHL account dashboard.

3. Monitor shipment updates: Keep an eye on real-time updates provided by DHL for the shipment's current location, estimated delivery time, and resolution of any issues.

4. Expedite shipping if needed: If the issue is causing a signifi

In [39]:
# Test Query 3: Route Optimization
print("\n" + "=" * 60)
print("QUERY 3: Route Optimization & Cost Savings")
print("=" * 60)

query_3 = "We need to ship 150 packages from New York to Boston. What would be the best way to optimize this route and what cost savings can we achieve?"

print(f"\n❓ Query: {query_3}\n")

response = agent.invoke({
    "messages": [
        {"role": "user", "content": query_3}
    ]
})

final_message = response["messages"][-1]
print("🤖 Agent Response:")
print(final_message.content)


QUERY 3: Route Optimization & Cost Savings

❓ Query: We need to ship 150 packages from New York to Boston. What would be the best way to optimize this route and what cost savings can we achieve?

🤖 Agent Response:
 Based on the provided information, here's the optimized route and cost savings for shipping 150 packages from New York to Boston:

- Route: New York → Boston
- Shipments: 150
- Base Cost/Shipment: $5.99
- Discount Applied: 20% (assuming a bulk discount)
- Cost/Shipment After Discount: $4.79
- Total Savings: $179.70 (($5.99 - $4.79) * 150 = $179.70)


In [38]:
# Test Query 2: Shipment Tracking
print("\n" + "=" * 60)
print("QUERY 2: Shipment Status Tracking")
print("=" * 60)

query_2 = "Can you check the status of shipment DHL001 and tell me when it's expected to arrive?"

print(f"\n❓ Query: {query_2}\n")

response = agent.invoke({
    "messages": [
        {"role": "user", "content": query_2}
    ]
})

final_message = response["messages"][-1]
print("🤖 Agent Response:")
print(final_message.content)


QUERY 2: Shipment Status Tracking

❓ Query: Can you check the status of shipment DHL001 and tell me when it's expected to arrive?

🤖 Agent Response:
 The shipment DHL001 is currently in transit from New York to Boston. It weighs 2.5 lbs and the cost of shipping is $12.50. The estimated delivery date for this package is February 4, 2026 at 15:26.


In [37]:
# Test Query 1: General DHL Service Information
print("=" * 60)
print("QUERY 1: General Service Information")
print("=" * 60)

query_1 = "What are DHL's standard delivery times and how do they differ from express service?"

print(f"\n❓ Query: {query_1}\n")

response = agent.invoke({
    "messages": [
        {"role": "user", "content": query_1}
    ]
})

# Extract final response
final_message = response["messages"][-1]
print("🤖 Agent Response:")
print(final_message.content)

QUERY 1: General Service Information

❓ Query: What are DHL's standard delivery times and how do they differ from express service?

🤖 Agent Response:
 To get the accurate information about DHL's standard delivery times and express service differences, I would need to access the DHL knowledge base using the provided function `query_dhl_knowledge`. However, I don't have that functionality at the moment.

Here is an example of how you can use it:

```
result = query_dhl_knowledge({"query": "What are DHL's standard delivery times and how do they differ from express service?"})
print(result)
```

Please note that the above code is a hypothetical example since I don't have the actual implementation of the `query_dhl_knowledge` function. You will need to replace it with your own implementation to get real results.

I suggest reaching out to DHL or visiting their official website for up-to-date and accurate information about their delivery times and express services.


## Demo: Interactive Shipment Tracking & Optimization Queries

Below are example queries you can run to test the RAG system. Each query demonstrates different capabilities of the agent.